# Follow-up Adequacy
**We examine censoring rates by diagnosis year to assess whether patients have sufficient follow-up to produce reliable pseudo-observations at 2 years.**

In [1]:
import numpy as np
import pandas as pd

## Import data

In [2]:
dtype_map = pd.read_csv('../outputs/pembro_carbo_features_dtypes.csv', index_col = 0).iloc[:, 0].to_dict()
df = pd.read_csv('../outputs/pembro_carbo_features_df.csv', dtype = dtype_map)

In [3]:
df.shape

(3706, 162)

## Censoring rates by years

In [4]:
df.groupby('adv_diagnosis_year')['event'].apply(lambda x: (x == 0).mean())

adv_diagnosis_year
2011    0.157025
2012    0.167665
2013    0.111702
2014    0.190000
2015    0.201550
2016    0.209924
2017    0.254777
2018    0.178161
2019    0.282353
2020    0.290698
2021    0.360000
2022    0.469543
2023    0.683908
Name: event, dtype: float64

In [5]:
(df.query('adv_diagnosis_year >= 2022')['event'] == 0).mean()

np.float64(0.5352112676056338)

In [6]:
results = []
for year in sorted(df['adv_diagnosis_year'].unique()):
    censored = df.query('adv_diagnosis_year == @year and event == 0', engine='python')
    if len(censored) > 0:
        frac = (censored['duration'] < 730).mean()
        results.append({'year': year, 'frac_censored_before_2y': frac})

pd.DataFrame(results)

,year,frac_censored_before_2y
0,2011,0.473684
1,2012,0.642857
2,2013,0.476190
3,2014,0.736842
4,2015,0.557692
5,2016,0.436364
6,2017,0.362500
7,2018,0.290323
8,2019,0.400000
9,2020,0.408000


**Among patients diagnosed with advanced disease in 2022 and later, >50% are censored and all lack 2-year follow-up from the data cut, making 2-year RMST pseudo-observations entirely extrapolated for these patients. The primary analysis is therefore restricted to patients diagnosed in 2021 or earlier, where the majority of patients have reached the 2-year endpoint and pseudo-observations are grounded in observed rather than extrapolated survival.**